In [ ]:
!pip install crewai langchain-google-genai supabase langsmith

In [ ]:
from google.colab import userdata, drive, files
import os

# 1.讀取自己的 Gemini API KEY
os.environ["GOOGLE_API_KEY"] = userdata.get('API_KEY_Three')

# 2.設置 LangSmith
os.environ["LANGCHAIN_TRACING_V2"] = "true"
os.environ["LANGCHAIN_ENDPOINT"] = "https://api.smith.langchain.com"
# 讀取自己的 LangSmith API key
os.environ["LANGCHAIN_API_KEY"] = userdata.get("LangSmith_API_Key")
# 這邊"LANGCHAIN_PROJECT"名稱會顯示在你的langSmith上
os.environ["LANGCHAIN_PROJECT"] = "T1_Career_pilot_project"

# 3.設置 Supabase
SUPABASE_URL = userdata.get('SUPABASE_URL')
SUPABASE_KEY = userdata.get('SUPABASE_SERVICE_ROLE_KEY')

In [ ]:
import os
from enum import Enum
from typing import List, Dict, Any, Optional
from pydantic import BaseModel, Field
from crewai import Agent, Task, Crew, Process
from crewai.tools import BaseTool
from langchain_google_genai import ChatGoogleGenerativeAI
import json
import time
from supabase import create_client, Client
from langsmith import traceable

In [ ]:
# --- 1. 模擬職缺資料 (Job Data) ---
# 這些是你未來爬蟲需要爬取的欄位
MOCK_JOBS = [
    {
        "id": "101",
        "title": "Python 後端工程師",
        "company": "台積電",
        "skills": ["Python", "Django", "MySQL", "Docker"],
        "description": "負責半導體製程資料分析平台開發，需熟悉 RESTful API 設計，具備雲端部屬經驗尤佳。",
        "salary": "60k-90k"
    },
    {
        "id": "102",
        "title": "AI 應用工程師",
        "company": "Google Taiwan",
        "skills": ["Python", "LangChain", "TensorFlow", "NLP"],
        "description": "開發基於 LLM 的企業級應用，需熟悉 RAG 架構、Prompt Engineering，並有實際專案落地經驗。",
        "salary": "100k-150k"
    },
    {
        "id": "103",
        "title": "前端工程師",
        "company": "LINE",
        "skills": ["React", "TypeScript", "Next.js", "CSS"],
        "description": "負責通訊軟體前端介面優化，需重視使用者體驗 (UX) 與效能調校。",
        "salary": "70k-100k"
    },
    {
        "id": "104",
        "title": "資料工程師",
        "company": "國泰金控",
        "skills": ["Python", "SQL", "ETL", "Airflow"],
        "description": "建置金融數據中台，負責資料清洗、ETL 流程自動化設計。",
        "salary": "65k-95k"
    }
]

# --- 2. 模擬使用者履歷 (User Resume) ---
MOCK_RESUME = {
  "user_id": "mock_user_001",
  "profile": {
    "name": "陳轉職",
    "education": [
      {
        "level": "University",
        "major": "企業管理系",
        "school_type": "Private",
        "status": "Graduated"
      },
      {
        "level": "High School",
        "major": "普通科",
        "school_type": "Public",
        "status": "Graduated"
      }
    ],
    "work_experience": [
      {
        "title": "數位行銷專員",
        "duration": "3 years",
        "description": "負責 FB 廣告投放，使用 Excel 進行數據分析，ROAS 提升 20%。"
      },
      {
        "title": "行政助理",
        "duration": "1 year",
        "description": "處理公司文書流程，影印文件與會議記錄。"
      }
    ],
    "skills": {
      "programming_languages": ["Python (自學3個月)"],
      "tools": ["Excel (VLOOKUP, Pivot Table)"],
      "languages": ["English (Reading/Writing: Medium)"]
    },
    "portfolio": [
      {
        "platform": "GitHub",
        "project_name": "Stock Crawler",
        "type": "Single .py file",
        "has_documentation": "false"
      }
    ],
    "autobiography": "您好，我是陳轉職，畢業於企管系。畢業後的第一份工作是行政助理，在那一年裡我學會了如何細心處理公司的各種雜物與文書流程，雖然工作穩定，但我總覺得希望能學到更多專業技能。後來轉職到電商公司擔任數位行銷專員，這三年的時間裡，我主要負責 Facebook 廣告的投放與監控。為了看廣告效果，我每天都要花很多時間操作 Excel，也是在這個時候，我發現自己對處理數據很有興趣，透過 VLOOKUP 和樞紐分析，我成功讓廣告的 ROAS 提升了 20%。\n\n在操作 Excel 的過程中，我聽說 Python 可以自動化處理這些重複的工作，於是三個月前開始利用下班時間在網路上自學。我目前已經學會了基礎的 Python 語法，並參考網路上的教學寫出了一個簡單的爬蟲腳本，可以用來抓取基本的股票資訊，這讓我很有成就感。雖然我目前的專案只有一個 .py 檔，也沒有寫什麼說明文件，但我對寫程式這件事非常有熱忱。\n\n我知道自己並非資工本科系畢業，技術力與資深工程師相比還有很大的落差，對於很多軟體開發的專業術語也還在摸索中。但我非常願意學習，也希望能找到一個願意給予轉職者機會的公司。我具備行銷的邏輯思維，也擅長溝通協調，希望能從初級助理工程師或實習職位開始做起，將過去的行銷經驗與現在學習的程式技術結合，在未來成為一名合格的軟體開發者。"
  }
}

# --- 3. 模擬職涯偏好問卷 (User Preferences) ---
MOCK_PREFERENCES = {
    "target_role": "後端工程師",
    "interested_industry": "科技業, 金融業",
    "salary_expectation": "50k 以上",
    "learning_goals": "想學習軟體應用開發",
    "motivation": "覺得行銷取代性高，想要有一技之長，對程式邏輯有興趣。"
}

# --- 4. 模擬履歷分析 (Resume Analysis) ---
MOCK_RESUME_ANALYSIS = {'overall_strengths': ['過去行銷經驗中，ROAS 提升 20% 的量化成果，展現數據分析與解決問題能力。', '具備行銷的邏輯思維和溝通協調能力。'], 'overall_weaknesses': ['履歷重心仍偏向過去的行銷與行政經驗，與目標「後端工程師」職位核心技能不符。', 'Python 自學經驗缺乏深度與應用場景的具體描述。', '作品集（Stock Crawler）缺乏文件說明，難以評估技術能力。', '針對「後端工程師」職位，履歷中的 ATS 關鍵字嚴重不足。'], 'critical_issues': [{'section': 'summary', 'original_text': '履歷的整體重心仍偏向過去的行銷與行政經驗，對於「後端工程師」這個目標職位而言，其核心技能與經驗的呈現不夠突出。技能部分僅列出Python（自學3個月），缺乏對學習深度、應用場景的具體描述。作品集僅提及一個單一檔案的爬蟲專案，且無文件說明。', 'issue_type': 'job_mismatch', 'issue_reason': '讓招募者難以快速評估其技術能力與潛力。', 'improvement_direction': '履歷開頭或摘要應更明確地指出轉職目標為「初級助理工程師」或「實習工程師」，而非直接鎖定「後端工程師」，以符合目前技能水平。強化「技能」區塊，即使是自學，也應列出已學習的Python函式庫、資料結構概念、或任何與後端開發相關的基礎知識（例如：資料庫基礎、API概念等），並說明學習程度。重新組織自傳，將轉職動機與學習過程精簡，並將更多篇幅用於強調已具備的程式基礎、解決問題的思維，以及對未來職位的理解與熱情。'}, {'section': 'projects', 'original_text': '在轉職目標「後端工程師」相關的技能與專案上，缺乏具體的成果與量化指標。例如，Python自學3個月，但未說明這三個月內完成了哪些具挑戰性的練習、解決了什麼問題、或專案的具體功能與效益。作品集中的「Stock Crawler」僅描述為「Single .py file」且「has_documentation: false」。', 'issue_type': 'lack_of_metrics', 'issue_reason': '無法有效證明其程式設計能力或專案管理能力。', 'improvement_direction': '針對Python自學部分，嘗試量化學習成果，例如：完成多少個練習專案、解決了哪些特定問題、或掌握了哪些特定技術點。強化「Stock Crawler」專案的描述，即使是單一檔案，也應說明其具體功能、使用的Python函式庫、資料來源、以及未來可能的擴展方向。若能加入程式碼行數、功能達成率等簡單指標會更好。考慮為專案添加最基礎的README文件，說明專案目的、如何運行、以及簡單的技術棧，以提升專業度。'}, {'section': 'skills', 'original_text': '針對「後端工程師」這個職位，履歷中的關鍵字嚴重不足。目前僅有「Python」一個直接相關的程式語言關鍵字，且未提及任何後端開發常用的框架（如Django, Flask）、資料庫（如SQL, MySQL, PostgreSQL）、版本控制工具（如Git）、雲端服務、API設計、資料結構、演算法等。', 'issue_type': 'keyword_missing', 'issue_reason': '這將導致履歷在ATS系統篩選時，很難被匹配到後端工程師的職位。', 'improvement_direction': '在「技能」區塊中，主動加入與後端開發相關的關鍵字，即使是基礎知識或正在學習中，例如：Git (版本控制), SQL (資料庫基礎), RESTful API (概念), Linux (基礎操作), 資料結構 (基礎概念)。若有接觸任何後端框架的基礎，即使是透過教學影片或線上課程，也應列出並註明學習程度。在自傳中，適度融入這些技術關鍵字，展現對後端領域的理解與學習意願。'}, {'section': 'overall', 'original_text': '使用者問卷中明確指出目標職位是「後端工程師」，但履歷內容與此目標存在顯著落差。目前的學經歷背景（企管系、數位行銷、行政助理）與技術技能（Python自學3個月、單一爬蟲專案）尚不足以直接應徵「後端工程師」職位。', 'issue_type': 'job_mismatch', 'issue_reason': '履歷整體呈現的專業度與技術深度仍需大幅提升，才能說服企業給予轉職者機會。', 'improvement_direction': '調整履歷的目標職位描述，更明確地鎖定「初級助理工程師」、「軟體開發實習生」或「程式開發學徒」等入門級職位，以提高被篩選的機會。積極參與線上課程、程式訓練營或開源專案，累積更多與後端開發相關的實作經驗與作品，並將這些經驗具體呈現在履歷中。考慮將過去的行銷經驗與程式技能結合，例如：開發一個自動化行銷數據分析工具，或一個小型電商後台管理系統，以展現跨領域的應用能力。針對目標職位，研究其常見的技術要求，並有針對性地學習和補充相關技能，例如：學習一個主流的後端框架（如Flask或Django的基礎）、資料庫操作、API設計與實作等。'}], 'ats_risk_level': 'high'}

# --- 5. 模擬缺口分析 (Skill Gap Analysis) ---
MOCK_SKILL_GAP_ANALYSIS = {'strengths': ['數據分析與商業邏輯思維', '強烈自主學習動機與實踐力', '跨領域溝通與協調能力'], 'weaknesses': ['缺乏核心後端技術棧實戰經驗 (如後端框架、API 設計、資料庫、雲端服務、容器化技術)', '缺乏版本控制系統 (Git) 協作經驗', '缺乏軟體測試與部署概念', '缺乏軟體開發生命週期 (SDLC) 實務經驗', '缺乏團隊協作開發經驗', '缺乏後端工程師職位常見關鍵字 (如後端框架、資料庫、雲端服務等)'], 'match_score': 25}

# --- 6. 模擬使用者優化後的履歷 (Optimization Resume) ---
MOCK_OPTIMIZATION_RESUME = {'professional_summary': '具備三年數位行銷經驗，擅長數據分析與廣告成效優化，成功提升廣告投資報酬率 (ROAS) 20%。對程式設計充滿熱情，自學 Python 並完成股票資訊爬蟲專案，渴望將行銷邏輯思維與程式技能結合，轉職成為初級助理工程師或軟體開發實習生。', 'professional_experience': ['數位行銷專員 (3 年) - 負責 Facebook 廣告的策略規劃與投放，並透過 Excel 進行廣告數據的深入分析與成效追蹤。成功將廣告投資報酬率 (ROAS) 提升 20%，有效優化行銷預算效益。', '行政助理 (1 年) - 負責處理公司日常行政事務與文書流程，包含文件歸檔、資料整理、會議記錄撰寫與跨部門協調。確保辦公室運作順暢，提升行政效率。'], 'core_skills': ['Python', 'Excel', '數據分析', 'Git', 'SQL', 'RESTful API'], 'projects': ['股票資訊爬蟲 (Stock Crawler): 基於 Python 開發，自動化抓取特定網站的股票基本資訊，展現基礎網頁資料擷取與處理能力。未來規劃加入資料儲存與視覺化功能。'], 'education': ['大學：企業管理學系 (私立大學，已畢業)', '高中：普通科 (公立高中，已畢業)'], 'autobiography': '您好，我是陳轉職，畢業於企業管理學系。我的職涯始於行政助理，在那一年裡，我學會了如何細心處理公司的各項行政事務與文書流程，並確保工作效率。雖然工作穩定，但我總希望能學習更多專業技能。隨後，我轉職至電商公司擔任數位行銷專員，在這三年的時間裡，我主要負責 Facebook 廣告的策略規劃與投放。為了精準監控廣告成效，我每天都投入大量時間操作 Excel 進行數據分析，也正是在這個過程中，我發現自己對處理數據與從中挖掘洞察有著濃厚的興趣。透過 VLOOKUP 和樞紐分析等工具，我成功將廣告的 ROAS 提升了 20%，有效優化了行銷預算效益。\n\n在深入數據分析的過程中，我了解到 Python 在自動化處理重複性工作上的強大潛力，這激發了我對程式設計的興趣。因此，三個月前我開始利用下班時間在網路上自學 Python，目前已掌握了基礎語法、資料結構與常用函式庫的應用。我也參考網路教學，獨立完成了一個簡單的股票資訊爬蟲專案，能自動抓取特定網站的基本股票數據，這讓我非常有成就感。在學習過程中，我也開始接觸到版本控制工具 Git 的基礎概念，並對資料庫的運作原理與 RESTful API 的設計概念產生了好奇。雖然目前專案仍是單一檔案且文件尚不完善，但我對程式開發的熱忱與學習動力非常高。\n\n我深知自己並非資訊工程本科系出身，目前在技術深度上與資深工程師仍有差距，對於許多軟體開發的專業術語也持續在學習與摸索中。然而，我具備極高的學習意願與解決問題的熱情，並渴望能將過去在行銷領域培養的邏輯思維與溝通協調能力，應用於程式開發。我希望能從初級助理工程師或軟體開發實習生的職位開始，在一個願意給予轉職者機會的環境中，持續精進我的程式技能，並將行銷實務經驗與程式技術結合，逐步成長為一名合格且具備跨領域視野的軟體開發者。'}

# --- 7. 模擬工作推薦清單 (Jobs_Recommendation) ---
MOCK_JOBS_RECOMMENDATION = [{
        "id": 1,
        "title": "資料工程師",
        "company": "國泰金控",
        "description": "建置金融數據中台，負責資料清洗、ETL 流程自動化設計。"
    },
    {
        "id": 2,
        "title": "AI 應用工程師",
        "company": "Google Taiwan",
        "description": "開發基於 LLM 的企業級應用，需熟悉 RAG 架構、Prompt Engineering，並有實際專案落地經驗。"
    },
    {
        "id": 3,
        "title": "Python 後端工程師",
        "company": "台積電",
        "description": "負責半導體製程資料分析平台開發，需熟悉 RESTful API 設計，具備雲端部屬經驗尤佳。"
    }]


In [ ]:
# ======================================================
# 1. Pydantic 資料模型定義 (Schema Layer)
# ======================================================

# --- 你的原始模型 (保持不變) ---
class JobItem(BaseModel):
  title: str = Field(description="職位名稱")
  company: str = Field(description="公司名稱")
  description: str = Field(description="職位描述摘要")
  match_reason: str = Field(description="推薦理由，為何這個職缺適合使用者")

class ResumeIssue(BaseModel):
  section: str = Field(description="履歷區塊名稱，如: 簡介、技能專長、專案、經歷、自傳")
  original_text: str = Field(description="使用者履歷中的原始文字內容，僅作為分析依據，不做任何評論說明，禁止修改")
  issue_type: List[str] = Field(description="問題類型分類（如：描述模糊、缺乏量化證據、ATS 關鍵字缺失、與目標職位不一致）,並詳加說明")
  severity: List[str] = Field(description="從企業篩選視角評估的嚴重程度(如：可優化、明顯扣分、直接刷掉、不修基本不用投),並詳加說明")
  diagnosis_dimension: str = Field(description="此問題主要影響的企業診斷面向")
  issue_reason: str = Field(description="站在企業 / HR / ATS 角度，說明為何此問題會降低錄取率")
  improvement_direction: List[str] = Field(description="可執行的改善方向，列點式說明（只說『該補什麼證據或結構』，不代寫內容）")

class ResumeAnalysis(BaseModel):
  candidate_positioning: str = Field(description="說明企業視角下，這份履歷目前『看起來像什麼角色』")
  target_role_gap_summary: str = Field(description="與目標職位（如後端工程師）之間的整體落差說明")
  overall_strengths: List[str] = Field(description="履歷中目前最具說服力、可保留的優勢點")
  overall_weaknesses: List[str] = Field(description="整體最影響錄取率的核心弱點")
  critical_issues: List[ResumeIssue] = Field(description="需要優先修正的關鍵問題清單（依嚴重度排序）")
  ats_risk_level: str = Field(description="從 ATS 與第一輪篩選角度評估的整體風險等級(如:低/中/高)")
  screening_outcome_prediction: str = Field(description="模擬企業 6–10 秒快速掃描後，最可能的篩選結果與原因")
  recommended_next_actions: List[str] = Field(description="不涉及代寫的前提下，給候選人的下一步行動建議，列點式說明")

class ResumeOptimization(BaseModel):
  professional_summary: str = Field(description="精簡的專業總結，需包含核心價值與推薦職缺的關鍵字")
  professional_experience: Optional[List[str]] = Field(default_factory=list, description="優化後的經歷列表。每筆包含 company, title, duration, 並以 STAR 原則重新撰寫的 description（條列式）")
  core_skills: List[str] = Field(description="從履歷中萃取與推薦職缺相關的技術或軟實力關鍵字6個")
  projects: Optional[List[str]] = Field(default_factory=list, description="優化後的專案描述，強調技術棧與量化成果")
  education: List[str] = Field(description="最高及次高學歷資訊列表，包含學校、學系、學位與畢業時間")
  autobiography: str = Field(description="保留使用者原本風格、敘事順序與用詞習慣前提下的優化後完整自傳")

class SkillAnalysis(BaseModel):
  strengths: List[str] = Field(description="與推薦職缺相關的使用者的核心優勢列表")
  weaknesses: List[str] = Field(description="與推薦職缺相關的能力缺口與弱點列表")
  match_score: int = Field(description="與推薦職缺的匹配度評分 (0-100)")

class ProjectPhase(BaseModel):
  phase_name: str = Field(description="階段名稱，例如：'Phase 1: 核心 API 與資料庫設計 (MVP Backend)'")
  phase_goal: str = Field(description="此階段的核心目標，說明為何要做這個階段")
  tasks: List[str] = Field(description="此階段需完成的具體任務清單（可直接作為 checklist 使用）")
  resume_value: str = Field(description="此階段完成後，可直接寫進履歷的一段敘述（偏向職缺導向關鍵字）")

class SideProject(BaseModel):
  project_name: str = Field(description="專案名稱。需具專業感，能清楚體現應用核心價值（例如：'基於 RAG 的法律文件自動審核系統' 而非 '法律 AI'）")
  capability_gaps_addressed: List[str] = Field(description="此專案主要補強的能力缺口清單（對應求職弱項）")
  tech_stack: List[str] = Field(description="完整技術棧清單，包含後端、資料庫、部署、容器化等")
  project_phases: List[ProjectPhase] = Field(description="專案分階段實作規劃，每一階段需包含目標、任務與履歷價值")
  overall_resume_impact: str = Field(description="整個專案完成後，對履歷競爭力的整體提升說明")
  difficulty: str = Field(description="專案整體難度與時程評估，格式為：'難度等級 (低/中/高) | 預估開發週期（含部署與測試）'，並簡述主要挑戰點。")

class CourseRecommendation(BaseModel):
  course_name: str = Field(description="課程名稱")
  platform: str = Field(description="推薦平台，如 Coursera, Udemy")
  duration: str = Field(description="預計學習時數")
  difficulty: str = Field(description="難度等級：入門 / 進階 / 專家")
  why_recommend: str = Field(description="推薦原因，需具體說明此課程如何彌補使用者的技能缺口")
  learning_points: List[str] = Field(description="此課程的 3 個核心學習亮點")

class CoverLetter(BaseModel):
  subject: str = Field(description="一個吸引人且專業的郵件主旨")
  content: str = Field(description="依據工具調用後產生的結果整理成完整的求職信內容")

# class FinalCareerReport(BaseModel):
#   """這就是我們最終希望 Agent 輸出的 JSON 結構"""
#   recommended_jobs: List[JobItem] = Field(default_factory=list, description="推薦的3個職缺列表")
#   resume_analysis: Optional[ResumeAnalysis] = Field(default=None, description="履歷分析的具體內容")
#   resume_optimization: Optional[ResumeOptimization] = Field(default=None, description="履歷優化的具體內容")
#   skill_gap_analysis: Optional[SkillAnalysis] = Field(default=None, description="技能缺口分析結果")
#   side_projects: Optional[SideProject] = Field(default=None, description="推薦的1個 Side Projects")
#   recommended_courses: List[CourseRecommendation] = Field(default_factory=list, description="推薦的3個學習課程列表")
#   cover_letter: List[CoverLetter] = Field(default_factory=list, description="依據推薦的不同職缺內容生成不同的客自化求職推薦信")

In [ ]:
# ======================================================
# 2. 工具與資料庫抓取層 (Tools Layer)
# ======================================================

class DatabaseTools:
  """不同功能的資料獲取工具"""

  @staticmethod
  def get_user_resume(user_id: str):
    """
    根據 user_id 到 Supabase 抓取用戶的履歷。
    """
    supabase = create_client(SUPABASE_URL, SUPABASE_KEY)

    try:
      # 執行 SQL 查詢
      response = supabase.table("resume") \
          .select("normalized_data") \
          .eq("user_id", user_id) \
          .single() \
          .execute()

      if not response.data:
        return {"error": "找不到該用戶履歷資料"}

      return response.data

    except Exception as e:
      return {"error": f"資料庫抓取失敗: {str(e)}"}

  @staticmethod
  def get_job_recommendation_profile(job_id: str):
    """
    到 Supabase 抓取推薦職缺資料，進行推薦信生成。
    """
    supabase = create_client(SUPABASE_URL, SUPABASE_KEY)

    try:
      # 執行 SQL 查詢
      response = supabase.table("job_posting") \
          .select("job_id, job_title, job_description, requirements") \
          .eq("job_id", job_id) \
          .single() \
          .execute()

      if not response.data:
        return {"error": "找不到職缺資料"}

      return response.data

    except Exception as e:
      return {"error": f"資料庫抓取失敗: {str(e)}"}

  @staticmethod
  def get_user_survey(user_id: str):
    """
    根據 user_id 到 Supabase 抓取用戶的問卷調查。
    """
    supabase = create_client(SUPABASE_URL, SUPABASE_KEY)

    try:
      # 執行 SQL 查詢
      response = supabase.table("career_survey") \
          .select("column, column, column") \
          .eq("user_id", user_id) \
          .single() \
          .execute()

      if not response.data:
        return {"error": "找不到該用戶問卷資料"}

      return response.data

    except Exception as e:
      return {"error": f"資料庫抓取失敗: {str(e)}"}

  @staticmethod
  def get_resume_analysis(user_id: str):
    """
    根據 user_id 到 Supabase 抓取用戶的履歷分析報告。
    """
    supabase = create_client(SUPABASE_URL, SUPABASE_KEY)

    try:
      # 執行 SQL 查詢
      response = supabase.table("resume_analysis") \
          .select("column, column, column") \
          .eq("user_id", user_id) \
          .single() \
          .execute()

      if not response.data:
        return {"error": "找不到該用戶問卷資料"}

      return response.data

    except Exception as e:
      return {"error": f"資料庫抓取失敗: {str(e)}"}

  @staticmethod
  def get_gap_analysis(user_id: str):
    """
    根據 user_id 到 Supabase 抓取用戶的缺口分析報告。
    """
    supabase = create_client(SUPABASE_URL, SUPABASE_KEY)

    try:
      # 執行 SQL 查詢
      response = supabase.table("career_analysis_report") \
          .select("column, column, column") \
          .eq("user_id", user_id) \
          .single() \
          .execute()

      if not response.data:
        return {"error": "找不到該用戶問卷資料"}

      return response.data

    except Exception as e:
      return {"error": f"資料庫抓取失敗: {str(e)}"}

  @staticmethod
  def get_optimize_resume(user_id: str):
    """
    根據 user_id 到 Supabase 抓取用戶的優化後履歷。
    """
    supabase = create_client(SUPABASE_URL, SUPABASE_KEY)

    try:
      # 執行 SQL 查詢
      response = supabase.table("optimize_resume") \
          .select("column, column, column") \
          .eq("user_id", user_id) \
          .single() \
          .execute()

      if not response.data:
        return {"error": "找不到該用戶問卷資料"}

      return response.data

    except Exception as e:
      return {"error": f"資料庫抓取失敗: {str(e)}"}

# 實作 CrewAI BaseTool
class FetchResumeTool(BaseTool):
  name: str = "FetchUserResume"
  description: str = "獲取使用者個人履歷。Input: user_id"
  def _run(self, user_id: str) -> str:
    return DatabaseTools.get_user_resume(user_id)

class RecommendJobSearchTool(BaseTool):
  name: str = "SearchRecommendJob"
  description: str = "搜尋推薦職缺。Input: job_id"
  def _run(self, job_id: str) -> str:
    return DatabaseTools.get_job_recommendation_profile(job_id)

class FetchSurveyTool(BaseTool):
  name: str = "FetchUserSurvey"
  description: str = "獲取使用者個人問卷結果。Input: user_id"
  def _run(self, user_id: str) -> str:
    return DatabaseTools.get_user_survey(user_id)

class FetchResumeAnalysisTool(BaseTool):
  name: str = "FetchUserResumeAnalysis"
  description: str = "獲取使用者個人履歷分析結果。Input: user_id"
  def _run(self, user_id: str) -> str:
    return DatabaseTools.get_resume_analysis(user_id)

class FetchGapAnalysisTool(BaseTool):
  name: str = "FetchUserGapAnalysis"
  description: str = "獲取使用者個人缺口分析結果。Input: user_id"
  def _run(self, user_id: str) -> str:
    return DatabaseTools.get_gap_analysis(user_id)

class FetchOptimizeResumeTool(BaseTool):
  name: str = "FetchUserOptimizeResume"
  description: str = "獲取使用者個人優化後的履歷。Input: user_id"
  def _run(self, user_id: str) -> str:
    return DatabaseTools.get_optimize_resume(user_id)

In [ ]:
# ======================================================
# 3. 核心配置管理 (Configuration Layer)
# ======================================================

class TaskType(Enum):
  JOB_REC = "job_rec"
  GAP_ANALYSIS = "gap_analysis"
  RESUME_ANALYSIS = "resume_analysis"
  RESUME_OPT = "resume_opt"
  COURSE_REC = "course_rec"
  PROJECT_REC = "project_rec"
  COVER_LETTER = "cover_letter"

class ProcessConfig:
  """
  這是一個中央廚房的菜單。
  定義每種任務需要什麼 Agent、什麼 Tool、以及最後要吐出什麼格式 (Pydantic)。
  """

  # 預先實例化工具
  tools_map = {
    "resume": FetchResumeTool(),
    "job_recommendation": RecommendJobSearchTool(),
    "survey": FetchSurveyTool(),
    "resume_analysis": FetchResumeAnalysisTool(),
    "gap_analysis": FetchGapAnalysisTool(),
    "optimize_resume": FetchOptimizeResumeTool(),
  }

  @classmethod
  def get_config(cls, task_type: TaskType):
    # 這裡根據 task_type 回傳動態配置
    configs = {
      TaskType.JOB_REC: {
        "agent_role": "職缺媒合專家",
        "agent_goal": "為使用者找到最匹配的市場職缺",
        "agent_backstory": "",
        # "tools": [cls.tools_map["resume"], cls.tools_map["survey"]],
        "output_model": JobItem, # 這裡指定結構
        "task_description": "分析 user_id='{user_id}' 的資料，並參考情境 '{context}'，找出適合的職缺。",
        "expected_output": "一份純文字的職缺清單"
      },
      TaskType.RESUME_ANALYSIS: {
        "agent_role": "服務過跨國科技公司（FAANG 等級）的資深招募顧問",
        "agent_goal": "針對使用者的問卷與原始履歷進行「第一輪履歷篩選與風險評估」，找出影響進入面試的核心問題",
        "agent_backstory": "你是一位擁有多年頂尖科技公司招募經驗的專家，深知企業平均只花 6–10 秒掃描一份履歷。你的視角極其專業且冷峻，專門從企業端尋找潛在風險。你只評論「履歷中已出現的內容」，絕不推測或新增使用者未提供的經驗。你的診斷標準包含：清楚度、證據力（數據化）、ATS 關鍵字完整度、以及與目標職涯的一致性",
        # "tools": [cls.tools_map["resume"], cls.tools_map["survey"]],
        "output_model": ResumeAnalysis,
        "task_description": """
          身為 FAANG 等級的資深招募顧問，你的任務是執行極其嚴苛的「第一輪履歷篩選與風險評估」。請對輸入的 [使用者問卷]: {survey} 與 [使用者履歷]: {resume} 進行交叉比對。
          你必須模擬企業 HR 僅有 6–10 秒的掃描行為，針對以下維度進行深度診斷：
          1.清楚度（Clarity）：排版邏輯是否讓關鍵資訊（職稱、公司、年資）一眼可見？敘述是否過於冗長？
          2.證據力（Evidence & Metrics）：檢視每一項工作經歷，判斷其是否具備數據支撐（例如：提升了幾 %、節省了多少成本）。
          3.ATS 關鍵字完整度：判斷履歷中的硬實力關鍵字是否能通過自動化篩選系統。
          4.目標一致性：評估目前的經歷敘述，是否足以支撐使用者在問卷中所表達的職涯目標。
          執行準則（Strict Rules）：
          1.零推測原則：你只能分析履歷中「白紙黑字」寫出的內容，嚴禁腦補使用者可能具備但未寫出的經驗。
          2.風險標註：若有空白期或職涯跨度過大等可能被 HR 挑戰的點，必須明確指出。
          3.可執行建議：每一項診斷出的缺點，都必須配對一個具體的「修改動作」。
        """,
        "expected_output": "一份純文字的診斷分析報告，必須包含：【清楚度、證據力、關鍵字、一致性】的現狀分析、具體的風險警示、以及分點列出的可執行修改建議。全文嚴禁包含任何 Markdown 格式標記（如 #, *, -, [ ] 等）。"
      },
      TaskType.RESUME_OPT: {
        "agent_role": "服務獵頭與企業 HR 的資深履歷策略顧問",
        "agent_goal": "在不破壞使用者個人風格的前提下，根據診斷結果將履歷優化為「專業人士的表達方式」，產出一份完整的履歷。",
        "agent_backstory": "你是一位擅長微調職涯敘事的專家。你的核心思維不是「重寫」，而是「修飾」。你具備精準模仿使用者語氣與句型的能力（偏敘事則維持敘事，偏條列則維持條列），確保優化後的內容讓使用者本人覺得「這仍然像是我寫的」。你拒絕使用過度制式化、一看就像 AI 產生的商業用語。你嚴格遵守 STAR 原則，若原履歷有數據則強化，無數據則保守描述，絕不憑空捏造。",
        # "tools": [cls.tools_map["resume"], cls.tools_map["resume_analysis"]],
        "output_model": ResumeOptimization,
        "task_description": """
          你現在是一位資深履歷策略顧問。你的目標是將 [原始履歷內容]: {resume} 根據 [履歷診斷分析結果]: {resume_analysis} 進行優化，但核心挑戰在於：優化後的履歷必須讓使用者覺得「這就是我寫的」。
          執行步驟如下：
          1.風格建模：首先分析原履歷的語氣（是謙遜還是自信？是精煉條列還是敘事性強？），定義其寫作風格輪廓。
          2.微調優化：在不改變原本敘事順序與語氣的前提下，進行「補清楚、補具體、補專業」的微調。
          3.STAR 化處理：將工作經歷轉化為自然且不生硬的 STAR 原則敘述。若原稿有數據，請將數據放在最顯眼的位置；若無數據，請以具體的「成果產出」描述代替，絕不偽造數據。
          4.專業轉化：將口語化的描述轉化為符合科技業文化的專業術語，但避免一看就是 AI 產出的制式模板（Cliché）。
          執行準則（Strict Rules）：
          1.嚴格維持原有的句型結構（偏敘事則維持敘事，偏條列則維持條列）。
          2.嚴禁新增任何原履歷中不存在的技能、證照或專案經驗。
        """,
        "expected_output": "內容需包含兩部分：1. 使用者的寫作風格輪廓定義。2. 優化後的完整履歷全文。全文嚴禁包含任何 Markdown 格式標記。"
      },
      TaskType.GAP_ANALYSIS: {
        "agent_role": "技能缺口分析師",
        "agent_goal": "分析技能落差",
        "agent_backstory": "",
        # "tools": [cls.tools_map["profile"], cls.tools_map["market"]],
        "output_model": SkillAnalysis,
        "task_description": "分析 user_id='{user_id}' 與目標職位的差距。情境: {context}",
        "expected_output": "一份純文字的分析報告"
      },
      TaskType.PROJECT_REC: {
        "agent_role": "熟悉科技業招募標準的職涯策略顧問",
        "agent_goal": "根據能力缺口分析，推薦一個最具「履歷補強價值」且具備職場可解釋性的 Side Project。",
        "agent_backstory": "你深知企業不在乎「學過什麼」，只在乎「是否實際完成過相關任務」。你擅長將技術學習轉化為「工作型任務」。你設計的專案並非教學或玩具專案，而是能拆解為多個實作階段（Phase-based）的商業模型，每個階段都能直接對應到履歷上的亮點。你不會假設使用者具備缺口以外的技能，確保建議是可落地執行的。",
        # "tools": [cls.tools_map["gap_analysis"]],
        "output_model": SideProject,
        "task_description": """
          身為職涯策略顧問，你必須根據 [能力缺口分析結果]: {skill_gap_result}，設計一個具備「職場可解釋性」的實戰專案。
          這個專案不是為了學習，而是為了**「在履歷上證明能力」**。設計邏輯必須符合以下規則：
          1.MVP（最小可行性）起步：從最核心、最能證明該缺口能力的基礎功能開始。
          2.多階段擴充（Phase-based）：專案必須拆解為 3–5 個階段。每個階段必須明確定義：
            a.階段目標：要向面試官證明哪項特定能力？
            b.實作任務：具體要做哪些開發或分析動作？
            c.履歷價值（Resume Value）：此階段完成後，如何用專業語言將其寫入履歷中。
          3.技術棧對齊：使用的技術必須是該職位主流且使用者可上手的。
          執行準則（Strict Rules）：
          1.避開所有「教學型、練習型」的常見專案（如：To-do List, Weather App 等）。
          2.專案必須具備真實世界的商業邏輯，讓面試官有興趣追問細節。
        """,
        "expected_output": "專案完整計畫書，包含：專案名稱、對應能力缺口、技術棧、詳細的階段規劃（Phase/Goal/Tasks/Resume Value）、競爭力提升說明、實作難度評估。全文嚴禁包含任何 Markdown 格式標記。"
      },
      TaskType.COURSE_REC: {
        "agent_role": "教育顧問",
        "agent_goal": "推薦學習資源",
        "agent_backstory": "",
        # "tools": [cls.tools_map["profile"], cls.tools_map["course"]],
        "output_model": CourseRecommendation,
        "task_description": "為 user_id='{user_id}' 推薦課程以補強技能。情境: {context}",
        "expected_output": "一份純文字的推薦清單"
      },
      TaskType.COVER_LETTER: {
        "agent_role": "服務外商與大型科技公司的資深獵頭",
        "agent_goal": "根據優化後的履歷與職缺資訊，撰寫一封能通過招募方審核、提高點擊率的 Cover Letter。",
        "agent_backstory": "你是一位掌握招募心理學的專家，知道 HR 只關心：你是否理解職位、你能帶來什麼價值、你是否值得面談。你撰寫的推薦信對齊 JD 的核心痛點，使用履歷中的事實作為證據，摒棄所有空泛的情緒用語。你的語氣專業且精準，能根據不同的產業文化調整筆觸。",
        "tools": [cls.tools_map["job_recommendation"]],
        "output_model": CoverLetter,
        "task_description": """
          必需先使用 'SearchRecommendJob' 工具，傳入 job_id='{job_id}' 以獲取推薦職缺資訊。
          你是一位深諳招募心理的高階獵頭。請根據 [優化後的履歷]: {optimize_resume} 與 [推薦職缺資訊]，撰寫一封精確命中 HR 痛點的 Cover Letter。
          你的撰寫邏輯必須圍繞招募方的三大核心考量：
          1.職位理解：證明你對這家公司當前的技術挑戰或商業痛點有深入思考。
          2.價值主張（Value Proposition）：從優化後的履歷中提取最強大的「證據」，對標 JD 中的核心需求。
          3.行動呼籲（Call to Action）：展現自信但不失禮貌的專業態度。
          格式與輸出內容要求：
          你必須將下述輸出內容彙整成完整的求職信文字：
          - 主旨：需包含職位與使用者最強賣點。
          - 問候語。
          - 開場：連結公司文化與個人動機。
          - 核心價值：對標 JD 的關鍵成就。
          - 實績證明：用數據或事實支撐。
          - 結語與 CTA：展現面談期待。
          執行準則（Strict Rules）：
          語氣必須根據推薦職缺的公司文化進行調整（如：新創公司用更具熱情的語氣，外商金融則用更嚴謹專業的語氣）。
          嚴禁使用「I am a hardworking candidate」等空泛的形容詞。
        """,
        "expected_output": "按照上述格式排列完整的求職信純文字內容。全文嚴禁包含任何 Markdown 格式標記"
      }
    }
    return configs.get(task_type)

In [ ]:
# ======================================================
# 4. 統一入口管理器 (Manager Layer)
# ======================================================

class CareerAgentManager:
    def __init__(self):
        self.llm = ChatGoogleGenerativeAI(model="gemini-2.5-flash", temperature=0.1, stream_usage=True)

    @traceable(name="CrewAI_Full_Run")
    def run_task(self, task_type_str: str, user_input: dict):

        try:
            task_type = TaskType(task_type_str)
        except ValueError:
            raise ValueError(f"不支援的 task_type: {task_type_str}")

        config = ProcessConfig.get_config(task_type)

        if config is None:
            raise ValueError(f"找不到 task 設定: {task_type_str}")

        # --- A. 建立兩個 Agent ---

        # 1. 執行 Agent (Worker)
        worker_agent = Agent(
            role=config["agent_role"],
            goal=config["agent_goal"],
            backstory=config["agent_backstory"],
            tools=config["tools"],
            verbose=True,
            llm=self.llm
        )

        # 2. 監控 Agent (QA)
        qa_agent = Agent(
            role="資深品質監控官",
            goal="審核內容的準確性、專業度，並確保輸出完全符合指定的 JSON 結構。",
            backstory="你是一位極其挑剔的審核者，任何邏輯錯誤或格式不合都無法逃過你的眼睛。",
            verbose=True,
            llm=self.llm
        )

        # --- B. 建立兩個 Task (Sequential 順序執行) ---

        # Task 1: 初步分析 (不需要 pydantic，產出 Raw Text 即可)
        work_task = Task(
            description=config["task_description"],
            expected_output=config["expected_output"],
            agent=worker_agent
        )

        # Task 2: 品質保證與結構化 (這一步才掛載 output_pydantic)
        qa_task = Task(
            description="""
            審核上一個任務的產出結果。
            1. 檢查是否有幻覺或錯誤資訊。
            2. 確保語氣對求職者專業且有建設性。
            3. 將最終結果依照指定的 Pydantic 結構進行封裝。
            """,
            expected_output="最終審核通過的結構化 JSON 報告。",
            agent=qa_agent,
            context=[work_task], # 讓 QA 拿到 Worker 的結果
            output_pydantic=config["output_model"] # <--- 最終輸出由 QA 負責結構化
        )

        # --- C. 組建 Crew 並執行 ---
        crew = Crew(
            agents=[worker_agent, qa_agent],
            tasks=[work_task, qa_task],
            process=Process.sequential, # 確保先做 Worker 再做 QA
            verbose=True
        )

        result = crew.kickoff(inputs = user_input)
        return result.pydantic.model_dump()

In [ ]:
# ======================================================
# 4. 測試區塊 (Testing Block)
# ======================================================

if __name__ == "__main__":
    manager = CareerAgentManager()
    # user_input = {"resume": MOCK_RESUME, "survey": MOCK_PREFERENCES} # 測試1 - resume_analysis
    # user_input = {"resume": MOCK_RESUME, "resume_analysis": MOCK_RESUME_ANALYSIS} # 測試2 - resume_opt
    # user_input = {"skill_gap_result": MOCK_SKILL_GAP_ANALYSIS} # 測試3 - project_rec
    user_input = {"optimize_resume": MOCK_OPTIMIZATION_RESUME, "job_id": 46} # 測試4 - cover_letter

    print(">>> 啟動 Resume Analysis 測試 (含 QA 流程)...")

    try:
        final_result = manager.run_task(
            task_type_str="cover_letter",
            user_input=user_input,
        )
        print("\n[最終 QA 審核結果]:")
        print(json.dumps(final_result, indent=2, ensure_ascii=False))
    except Exception as e:
        print(f"測試失敗: {e}")

/tmp/ipython-input-1331565166.py:6: UserWarning: WARNING! stream_usage is not default parameter.
                stream_usage was transferred to model_kwargs.
                Please confirm that stream_usage is what you intended.
  manager = CareerAgentManager()


>>> 啟動 Resume Analysis 測試 (含 QA 流程)...


╭─────────────────────────────────────────── 🚀 Crew Execution Started ───────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Started                                                                                         │
│  Name:                                                                                                          │
│  crew                                                                                                           │
│  ID:                                                                                                            │
│  f8976a77-f9c9-49a4-9b78-a95da5a09cae                                                                           │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────────── 📋 Task Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Started                                                                                                   │
│  Name:                                                                                                          │
│            必需先使用 'SearchRecommendJob' 工具，傳入 job_id='46' 以獲取推薦職缺資訊。                          │
│            你是一位深諳招募心理的高階獵頭。請根據 [優化後的履歷]: {'professional_summary':                      │
│  '具備三年數位行銷經驗，擅長數據分析與廣告成效優化，成功提升廣告投資報酬率 (ROAS)                               │
│  20%。對程式設計充滿熱情，自學 Python                                                                           │
│  並完成股票資訊爬蟲專案，渴望將行銷邏輯思維與程式技能結合，轉職成為初級助理工程師或軟體開發實習生。',           │
│  'professional_experience': ['數位行銷專員 (3 年) - 負責 Facebook 廣告的策略規劃與投放，並透過 Excel            │
│  進行廣告數據的深入分析與成效追蹤。成功將廣告投資報酬率 (ROAS) 提升 20%，有效優化行銷預算效益。', '行政助理 (1  │
│  年) -                                                                                                          │
│  負責處理公司日常行政事務與文書流程，包含文件歸檔、資料整理、會議記錄撰寫與跨部門協調。確保辦公室運作順暢，提   │
│  升行政效率。'], 'core_skills': ['Python', 'Excel', '數據分析', 'Git', 'SQL', 'RESTful API'], 'projects':       │
│  ['股票資訊爬蟲 (Stock Crawler): 基於 Python                                                                    │
│  開發，自動化抓取特定網站的股票基本資訊，展現基礎網頁資料擷取與處理能力。未來規劃加入資料儲存與視覺化功能。'],  │
│  'education': ['大學：企業管理學系 (私立大學，已畢業)', '高中：普通科 (公立高中，已畢業)'], 'autobiography':    │
│  '您好，我是陳轉職，畢業於企業管理學系。我的職涯始於行政助理，在那一年裡，我學會了如何細心處理公司的各項行政事  │
│  務與文書流程，並確保工作效率。雖然工作穩定，但我總希望能學習更多專業技能。隨後，我轉職至電商公司擔任數位行銷   │
│  專員，在這三年的時間裡，我主要負責 Facebook                                                                    │
│  廣告的策略規劃與投放。為了精準監控廣告成效，我每天都投入大量時間操作 Excel                                     │
│  進行數據分析，也正是在這個過程中，我發現自己對處理數據與從中挖掘洞察有著濃厚的興趣。透過 VLOOKUP               │
│  和樞紐分析等工具，我成功將廣告的 ROAS 提升了                                                                   │
│  20%，有效優化了行銷預算效益。\n\n在深入數據分析的過程中，我了解到 Python                                       │
│  在自動化處理重複性工作上的強大潛力，這激發了我對程式設計的興趣。因此，三個月前我開始利用下班時間在網路上自學   │
│  Python，目前已掌握了基礎語法、資料結構與常用函式庫的應用。我也參考網路教學，獨立完成了一個簡單的股票資訊爬蟲   │
│  專案，能自動抓取特定網站的基本股票數據，這讓我非常有成就感。在學習過程中，我也開始接觸到版本控制工具 Git       │
│  的基礎概念，並對資料庫的運作原理與 RESTful API                                                                 │
│  的設計概念產生了好奇。雖然目前專案仍是單一檔案且文件尚不完善，但我對程式開發的熱忱與學習動力非常高。\n\n我深   │
│  知自己並非資訊工程本科系出身，目前在技術深度上與資深工程師仍有差距，對於許多軟體開發的專業術語也持續在學習與   │
│  摸索中。然而，我具備極高的學習意願與解決問題的熱情，並渴望能將過去在行銷領域培養的邏輯思維與溝通協調能力，應   │
│  用於程式開發。我希望能從初級助理工程師或軟體開發實習生的職位開始，在一個願意給予轉職者機會的環境中，持續精進   │
│  我的程式技能，並將行銷實務經驗與程式技術結合，逐步成長為一名合格且具備跨領域視野的軟體開發者。'} 與            │
│  [推薦職缺資訊]，撰寫一封精確命中 HR 痛點的 Cover Letter。                                                      │
│            你的撰寫邏輯必須圍繞招募方的三大核心考量：                                                           │
│            1.職位理解：證明你對這家公司當前的技術挑戰或商業痛點有深入思考。                                     │
│            2.價值主張（Value Proposition）：從優化後的履歷中提取最強大的「證據」，對標 JD 中的核心需求。        │
│            3.行動呼籲（Call to Action）：展現自信但不失禮貌的專業態度。                                         │
│            格式與輸出內容要求：                                                                                 │
│            你必須將下述輸出內容彙整成完整的求職信文字：                                                         │
│            - 主旨：需包含職位與使用者最強賣點。                                                                 │
│            - 問候語。                                                                                           │
│            - 開場：連結公司文化與個人動機。                                                                     │
│            - 核心價值：對標 JD 的關鍵成就。                                                                     │
│            - 實績證明：用數據或事實支撐。                                                                       │
│            - 結語與 CTA：展現面談期待。                           

╭─────────────────────────────────────────────── 🤖 Agent Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: 服務外商與大型科技公司的資深獵頭                                                                        │
│                                                                                                                 │
│  Task:                                                                                                          │
│            必需先使用 'SearchRecommendJob' 工具，傳入 job_id='46' 以獲取推薦職缺資訊。                          │
│            你是一位深諳招募心理的高階獵頭。請根據 [優化後的履歷]: {'professional_summary':                      │
│  '具備三年數位行銷經驗，擅長數據分析與廣告成效優化，成功提升廣告投資報酬率 (ROAS)                               │
│  20%。對程式設計充滿熱情，自學 Python                                                                           │
│  並完成股票資訊爬蟲專案，渴望將行銷邏輯思維與程式技能結合，轉職成為初級助理工程師或軟體開發實習生。',           │
│  'professional_experience': ['數位行銷專員 (3 年) - 負責 Facebook 廣告的策略規劃與投放，並透過 Excel            │
│  進行廣告數據的深入分析與成效追蹤。成功將廣告投資報酬率 (ROAS) 提升 20%，有效優化行銷預算效益。', '行政助理 (1  │
│  年) -                                                                                                          │
│  負責處理公司日常行政事務與文書流程，包含文件歸檔、資料整理、會議記錄撰寫與跨部門協調。確保辦公室運作順暢，提   │
│  升行政效率。'], 'core_skills': ['Python', 'Excel', '數據分析', 'Git', 'SQL', 'RESTful API'], 'projects':       │
│  ['股票資訊爬蟲 (Stock Crawler): 基於 Python                                                                    │
│  開發，自動化抓取特定網站的股票基本資訊，展現基礎網頁資料擷取與處理能力。未來規劃加入資料儲存與視覺化功能。'],  │
│  'education': ['大學：企業管理學系 (私立大學，已畢業)', '高中：普通科 (公立高中，已畢業)'], 'autobiography':    │
│  '您好，我是陳轉職，畢業於企業管理學系。我的職涯始於行政助理，在那一年裡，我學會了如何細心處理公司的各項行政事  │
│  務與文書流程，並確保工作效率。雖然工作穩定，但我總希望能學習更多專業技能。隨後，我轉職至電商公司擔任數位行銷   │
│  專員，在這三年的時間裡，我主要負責 Facebook                                                                    │
│  廣告的策略規劃與投放。為了精準監控廣告成效，我每天都投入大量時間操作 Excel                                     │
│  進行數據分析，也正是在這個過程中，我發現自己對處理數據與從中挖掘洞察有著濃厚的興趣。透過 VLOOKUP               │
│  和樞紐分析等工具，我成功將廣告的 ROAS 提升了                                                                   │
│  20%，有效優化了行銷預算效益。\n\n在深入數據分析的過程中，我了解到 Python                                       │
│  在自動化處理重複性工作上的強大潛力，這激發了我對程式設計的興趣。因此，三個月前我開始利用下班時間在網路上自學   │
│  Python，目前已掌握了基礎語法、資料結構與常用函式庫的應用。我也參考網路教學，獨立完成了一個簡單的股票資訊爬蟲   │
│  專案，能自動抓取特定網站的基本股票數據，這讓我非常有成就感。在學習過程中，我也開始接觸到版本控制工具 Git       │
│  的基礎概念，並對資料庫的運作原理與 RESTful API                                                                 │
│  的設計概念產生了好奇。雖然目前專案仍是單一檔案且文件尚不完善，但我對程式開發的熱忱與學習動力非常高。\n\n我深   │
│  知自己並非資訊工程本科系出身，目前在技術深度上與資深工程師仍有差距，對於許多軟體開發的專業術語也持續在學習與   │
│  摸索中。然而，我具備極高的學習意願與解決問題的熱情，並渴望能將過去在行銷領域培養的邏輯思維與溝通協調能力，應   │
│  用於程式開發。我希望能從初級助理工程師或軟體開發實習生的職位開始，在一個願意給予轉職者機會的環境中，持續精進   │
│  我的程式技能，並將行銷實務經驗與程式技術結合，逐步成長為一名合格且具備跨領域視野的軟體開發者。'} 與            │
│  [推薦職缺資訊]，撰寫一封精確命中 HR 痛點的 Cover Letter。                                                      │
│            你的撰寫邏輯必須圍繞招募方的三大核心考量：                                                           │
│            1.職位理解：證明你對這家公司當前的技術挑戰或商業痛點有深入思考。                                     │
│            2.價值主張（Value Proposition）：從優化後的履歷中提取最強大的「證據」，對標 JD 中的核心需求。        │
│            3.行動呼籲（Call to Action）：展現自信但不失禮貌的專業態度。                                         │
│            格式與輸出內容要求：                                                                                 │
│            你必須將下述輸出內容彙整成完整的求職信文字：                                                         │
│            - 主旨：需包含職位與使用者最強賣點。                                                                 │
│            - 問候語。                                                                                           │
│            - 開場：連結公司文化與個人動機。                                                                     │
│            - 核心價值：對標 JD 的關鍵成就。                                                                     │
│            - 實績證明：用數據或事實支撐。                              

...

╭─────────────────────────────────────────────── 🤖 Agent Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: 資深品質監控官                                                                                          │
│                                                                                                                 │
│  Task:                                                                                                          │
│              審核上一個任務的產出結果。                                                                         │
│              1. 檢查是否有幻覺或錯誤資訊。                                                                      │
│              2. 確保語氣對求職者專業且有建設性。                                                                │
│              3. 將最終結果依照指定的 Pydantic 結構進行封裝。                                                    │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭───────────────────────────────────────────── ✅ Agent Final Answer ─────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: 資深品質監控官                                                                                          │
│                                                                                                                 │
│  Final Answer:                                                                                                  │
│  ```json                                                                                                        │
│  {                                                                                                              │
│    "subject": "軟體設計工程師 - 具備數據分析思維與Python開發潛力之轉職者",                                      │
│    "content":                                                                                                   │
│  "您好，嚮樂科技招募團隊：\n\n我對貴公司在通訊設備軟硬體整合領域的創新與高速成長深感興趣，特別是貴公司正在開發  │
│  全新雲端工控設備及軟體的願景。我理解軟體設計工程師在其中扮演的關鍵角色，需要具備扎實的程式開發能力與解決問題   │
│  的熱情。\n\n我在數位行銷領域累積了三年經驗，擅長透過數據分析優化廣告成效，成功將廣告投資報酬率（ROAS）提升20%  │
│  。這段經歷培養了我嚴謹的邏輯思維與對細節的關注。基於對程式設計的熱情，我自學Python並完成股票資訊爬蟲專案，展   │
│  現了基礎網頁資料擷取與處理能力，並熟悉Git版本控制系統。雖然我目前是轉職者，但我具備快速學習新技術的能力，並渴  │
│  望將過去的分析思維與程式技能結合，為貴公司的工控產品軟體開發貢獻心力。\n\n我已具備Linux系統操作經驗，並能自行  │
│  研讀英文技術文件。我深信我的學習熱忱、責任感以及積極樂觀的團隊合作精神，能讓我快速融入貴公司年輕、熱情且創新   │
│  的工作團隊。\n\n期待能有機會進一步面談，分享我如何將行銷數據分析的經驗應用於軟體開發，並為嚮樂科技帶來實質價   │
│  值。\n\n陳轉職 敬上"                                                                                           │
│  }                                                                                                              │
│  ```                                                                                                            │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭────────────────────────────────────────────── 📋 Task Completion ───────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Completed                                                                                                 │
│  Name:                                                                                                          │
│                                                                                                                 │
│              審核上一個任務的產出結果。                                                                         │
│              1. 檢查是否有幻覺或錯誤資訊。                                                                      │
│              2. 確保語氣對求職者專業且有建設性。                                                                │
│              3. 將最終結果依照指定的 Pydantic 結構進行封裝。                                                    │
│                                                                                                                 │
│  Agent:                                                                                                         │
│  資深品質監控官                                                                                                 │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────────── Crew Completion ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Completed                                                                                       │
│  Name:                                                                                                          │
│  crew                                                                                                           │
│  ID:                                                                                                            │
│  f8976a77-f9c9-49a4-9b78-a95da5a09cae                                                                           │
│  Final Output: ```json                                                                                          │
│  {                                                                                                              │
│    "subject": "軟體設計工程師 - 具備數據分析思維與Python開發潛力之轉職者",                                      │
│    "content":                                                                                                   │
│  "您好，嚮樂科技招募團隊：\n\n我對貴公司在通訊設備軟硬體整合領域的創新與高速成長深感興趣，特別是貴公司正在開發  │
│  全新雲端工控設備及軟體的願景。我理解軟體設計工程師在其中扮演的關鍵角色，需要具備扎實的程式開發能力與解決問題   │
│  的熱情。\n\n我在數位行銷領域累積了三年經驗，擅長透過數據分析優化廣告成效，成功將廣告投資報酬率（ROAS）提升20%  │
│  。這段經歷培養了我嚴謹的邏輯思維與對細節的關注。基於對程式設計的熱情，我自學Python並完成股票資訊爬蟲專案，展   │
│  現了基礎網頁資料擷取與處理能力，並熟悉Git版本控制系統。雖然我目前是轉職者，但我具備快速學習新技術的能力，並渴  │
│  望將過去的分析思維與程式技能結合，為貴公司的工控產品軟體開發貢獻心力。\n\n我已具備Linux系統操作經驗，並能自行  │
│  研讀英文技術文件。我深信我的學習熱忱、責任感以及積極樂觀的團隊合作精神，能讓我快速融入貴公司年輕、熱情且創新   │
│  的工作團隊。\n\n期待能有機會進一步面談，分享我如何將行銷數據分析的經驗應用於軟體開發，並為嚮樂科技帶來實質價   │
│  值。\n\n陳轉職 敬上"                                                                                           │
│  }                                                                                                              │
│  ```                                                                                                            │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯


[最終 QA 審核結果]:
{
  "subject": "軟體設計工程師 - 具備數據分析思維與Python開發潛力之轉職者",
  "content": "您好，嚮樂科技招募團隊：\n\n我對貴公司在通訊設備軟硬體整合領域的創新與高速成長深感興趣，特別是貴公司正在開發全新雲端工控設備及軟體的願景。我理解軟體設計工程師在其中扮演的關鍵角色，需要具備扎實的程式開發能力與解決問題的熱情。\n\n我在數位行銷領域累積了三年經驗，擅長透過數據分析優化廣告成效，成功將廣告投資報酬率（ROAS）提升20%。這段經歷培養了我嚴謹的邏輯思維與對細節的關注。基於對程式設計的熱情，我自學Python並完成股票資訊爬蟲專案，展現了基礎網頁資料擷取與處理能力，並熟悉Git版本控制系統。雖然我目前是轉職者，但我具備快速學習新技術的能力，並渴望將過去的分析思維與程式技能結合，為貴公司的工控產品軟體開發貢獻心力。\n\n我已具備Linux系統操作經驗，並能自行研讀英文技術文件。我深信我的學習熱忱、責任感以及積極樂觀的團隊合作精神，能讓我快速融入貴公司年輕、熱情且創新的工作團隊。\n\n期待能有機會進一步面談，分享我如何將行銷數據分析的經驗應用於軟體開發，並為嚮樂科技帶來實質價值。\n\n陳轉職 敬上"
}


In [ ]:
print(json.dumps(final_result, indent=2, ensure_ascii=False)) # 測試1 - resume_analysis

{
  "candidate_positioning": "從企業角度來看，這份履歷目前呈現的是一位具備非技術職位經驗（數位行銷、行政助理），且對後端工程師職位有轉職意願的初學者。然而，由於缺乏相關的技術實務經驗、專案成果及關鍵字，目前被定位為與目標職位要求存在巨大落差，屬於高風險候選人。",
  "target_role_gap_summary": "此履歷與目標後端工程師職位之間存在顯著的整體落差。主要體現在缺乏直接相關的技術經驗、量化成果、關鍵字，以及作品集未能展現足夠的技術深度與複雜度。現有工作經驗多為非技術職位，導致企業難以評估其後端開發潛力，且在ATS篩選階段面臨極高風險。",
  "overall_strengths": [
    "履歷結構清晰，職稱與年資易於辨識。",
    "具備數位行銷專員經驗中「ROAS 提升 20%」的量化成果，展現商業思維與成果導向（儘管與目標職位關聯性低）。",
    "自傳中表達了明確的轉職意願和學習熱忱。"
  ],
  "overall_weaknesses": [
    "缺乏與後端工程師職位直接相關的技術經驗、專案成果及量化數據，無法證明實際開發能力。",
    "履歷中嚴重缺乏後端工程師職位所需的關鍵字，導致ATS篩選失敗率極高。",
    "作品集描述過於簡陋，未能有效展現技術深度與複雜度，甚至暴露不足。",
    "自傳內容冗長且包含自我貶低語句，影響專業形象與自信心。",
    "整體履歷內容（學歷、工作經驗、技能）與目標職位（後端工程師）的目標一致性極低。"
  ],
  "critical_issues": [
    {
      "section": "簡介/自傳",
      "original_text": "自傳內容過於冗長，包含大量個人動機、自我評估及重複資訊，稀釋了關鍵技能與成果的能見度。其中包含的自我貶低語句（例如「技術力與資深工程師相比還有很大的落差」）可能讓招募者對您的自信心和準備度產生疑慮。",
      "issue_type": [
        "內容冗餘",
        "語氣不專業",
        "描述模糊"
      ],
      "severity": [
        "直接刷掉",
        "不修基本不用投"


In [ ]:
print(json.dumps(final_result, indent=2, ensure_ascii=False)) # 測試2 - resume_opt

{
  "professional_summary": "陳轉職是一位具備三年數位行銷經驗的專業人士，擅長數據處理與邏輯分析，並成功將廣告ROAS提升20%。憑藉對程式設計的極高熱忱，自學Python並獨立開發股票爬蟲專案，展現強烈自我學習與跨領域轉型能力。具備行銷邏輯思維與良好溝通協調能力，對Git、SQL、RESTful API有初步理解。渴望從初級助理工程師或軟體開發實習生職位開始，將跨領域經驗與程式技術結合，逐步成長為軟體開發者。",
  "professional_experience": [
    "電商公司 | 數位行銷專員 | 3年 | 負責電商平台Facebook廣告投放與成效監控。運用 Excel 進行廣告數據深度分析，包含 VLOOKUP 與樞紐分析，識別高潛力受眾與優化投放策略，成功將廣告投資報酬率 (ROAS) 提升 20%。",
    "公司 | 行政助理 | 1年 | 負責公司日常行政事務與文書管理。建立並維護標準化文書處理流程，負責文件歸檔、資料整理及會議記錄，確保了營運流程的順暢與資訊的準確傳達。"
  ],
  "core_skills": [
    "Python",
    "數據分析",
    "邏輯思維",
    "Git",
    "SQL",
    "自我學習能力"
  ],
  "projects": [
    "Stock Crawler (Python爬蟲專案)：利用Python開發單一.py檔案，成功抓取基礎股票資訊，展現Python基礎語法與資料結構應用能力。"
  ],
  "education": [
    "大學 | 企業管理系 | 學士 | 未提供",
    "高中 | 普通科 | 畢業 | 未提供"
  ],
  "autobiography": "您好，我是陳轉職，畢業於企業管理系。在職涯初期，我曾擔任行政助理一年，期間培養了細心處理各項事務與流程管理的嚴謹度。儘管工作穩定，我仍渴望學習更具專業性的技能。隨後轉職至電商公司擔任數位行銷專員三年，主要負責Facebook廣告的投放與監控。為了精準評估廣告成效，我投入大量時間操作 Excel 進行數據分析，並透過 VLOOKUP 和樞紐分析等工具，成功將廣告的 ROAS 提升了 20%。這段經歷讓我發現自己對數據處理與邏輯分析的濃厚

In [ ]:
print(json.dumps(final_result, indent=2, ensure_ascii=False)) # 測試3 - project_rec

{
  "project_name": "本地服務供應商目錄 API (Local Service Provider Directory API)",
  "capability_gaps_addressed": [
    "缺乏核心後端技術棧實戰經驗 (如後端框架、API 設計、資料庫、雲端服務、容器化技術)",
    "缺乏版本控制系統 (Git) 協作經驗",
    "缺乏軟體測試與部署概念",
    "缺乏軟體開發生命週期 (SDLC) 實務經驗",
    "缺乏團隊協作開發經驗 (透過結構化開發流程與 Git 實踐模擬)",
    "缺乏後端工程師職位常見關鍵字 (如後端框架、資料庫、雲端服務等)"
  ],
  "tech_stack": [
    "Python Flask",
    "PostgreSQL",
    "SQLAlchemy (搭配 Flask-SQLAlchemy)",
    "JSON Web Tokens (JWT)",
    "Git (GitHub)",
    "Docker",
    "Amazon Web Services (AWS Lightsail 或 EC2)",
    "Gunicorn",
    "Nginx",
    "Pytest"
  ],
  "project_phases": [
    {
      "phase_name": "Phase 1: 核心服務供應商與使用者管理 API (MVP)",
      "phase_goal": "建立基礎後端架構、資料庫互動與核心 API 端點。證明具備建構 RESTful API 並與資料庫互動的能力。",
      "tasks": [
        "初始化 Flask 專案結構，規劃模組化目錄。",
        "定義使用者 (User) 與服務供應商 (ServiceProvider) 的資料庫模型 (Model)，包含基本欄位如名稱、電子郵件、密碼雜湊、服務類別、描述、聯絡資訊等。",
        "實作使用者註冊與登入功能，並導入 JWT 進行身份驗證與授權。",
        "開發服務供應商資料的 CRUD (Create, Read, Update, Delete) API 端點。",
  

In [ ]:
print(json.dumps(final_result, indent=2, ensure_ascii=False)) # 測試4 - cover_letter

{
  "subject": "應徵軟體設計工程師 - 具備數據分析思維與Python開發實績的轉職者",
  "content": "您好，嚮樂科技招募團隊：\n\n我對貴公司在通訊設備軟硬體整合領域的創新與高速成長深感興趣，特別是貴公司正在尋找軟體設計工程師參與全新雲端工控設備及軟體開發的機會。我理解此職位需要具備embedded system軟體程式撰寫能力，並熟悉Linux系統操作與版本控制工具。\n\n我在數位行銷領域累積了三年數據分析與廣告成效優化的實務經驗，成功將廣告投資報酬率（ROAS）提升20%，這段經歷培養了我嚴謹的邏輯思維與解決問題的能力。同時，我對程式設計抱持高度熱情，自學Python並獨立完成了股票資訊爬蟲專案，展現了基礎網頁資料擷取與處理能力。我已掌握Python基礎語法、資料結構與常用函式庫，並具備Git版本控制系統的使用經驗，這與貴公司對版本控制系統經驗的需求相符。\n\n雖然我目前尚無C/C++開發經驗，但我具備快速學習新技術的能力與高度學習意願，並能自行研讀英文技術文件。我渴望將過去在行銷領域培養的數據分析思維與程式技能結合，貢獻於貴公司的軟體開發團隊。\n\n期待能有機會進一步面談，分享我如何將跨領域的經驗應用於軟體設計工程師的職位，並為嚮樂科技帶來價值。\n\n陳轉職 敬上"
}
